# IHP SG13G2 Octagon L2n0 — measured two-port comparison

Compare the **complete complex S-matrix** from the recorded Palace sweep with
both pinned, THRU-de-embedded measured `.s2p` files. The reference impedance is
50 Ω. The 11 Palace frequencies span 1–50 GHz; measured complex S-values are
interpolated onto those frequencies. Magnitude, phase and complex-error metrics
are reported for S11, S21, S12 and S22. These comparisons are the validation
criterion; an extracted inductance by itself cannot validate the device.

The IHP OpenEMS runtime was unavailable, so these comparisons use the pinned
measured S-parameter data. The Palace excitation uses interlayer ports and a
minimal reference pad, whereas the pinned OpenEMS script uses TopMetal1 edge
ports and a Metal1/common-ground pillar. The model also has unresolved mesh
error and does not yet reproduce the measured two-port response.

For diagnostic comparison with the IHP script, differential impedance is
`Z11 - Z12 - Z21 + Z22`. The earlier `2 Z0 (1-S21)/S21` result assumes a
pure series element and is retained only as a diagnostic; its 1 GHz agreement
is not evidence of model accuracy.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import skrf as rf

from gsim.common.benchmarks import ihp_l2n0 as b

plt.rcParams["figure.dpi"] = 120
RUN = b.RUN_ROOT / "l2n0_best"
run = b.load_recorded_run(RUN)
sp = run.sparams
print("Palace sweep:", sp.port_names, np.asarray(sp.freq), "GHz")

In [ ]:
def lq_from_s(s, f_hz, z0=b.Z0_OHM):
    z = 2.0 * z0 * (1.0 - s[1, 0]) / s[1, 0]
    l_nh = z.imag / (2 * np.pi * f_hz) * 1e9
    q = z.imag / z.real if z.real != 0 else float("nan")
    return l_nh, q, z


def palace_curves():
    rows = []
    for i, f_ghz in enumerate(np.asarray(sp.freq, dtype=float)):
        s = b.s_matrix(sp, i)
        l, q, z = b.lq_from_s_matrix(s, f_ghz * 1e9)
        rows.append((f_ghz, l, q, s))
    return rows


def measured_curves(label):
    n = rf.Network(str(b.MEASURED[label]))
    z0 = float(np.real(np.asarray(n.z0).reshape(-1)[0]))
    z = 2.0 * z0 * (1.0 - n.s[:, 1, 0]) / n.s[:, 1, 0]
    l = z.imag / (2 * np.pi * n.f) * 1e9
    q = np.divide(z.imag, z.real, out=np.full_like(z.real, np.nan), where=z.real != 0)
    return n.f / 1e9, l, q, n.s


palace = palace_curves()
measured = {label: measured_curves(label) for label in ("die_x2y7", "die_x5y8")}

In [ ]:
import json

experiments = b.REPO / "nbs" / "data" / "ihp_validation" / "experiments"
matrix = json.loads((experiments / "results.json").read_text())
verify = json.loads((experiments / "verification" / "verification.json").read_text())
print(f"{'model':9} {'f GHz':>6} {'L nH':>7} {'Q':>7} | {'x2y7 L':>7} {'x5y8 L':>7}")
for name in ("baseline", "best"):
    for f_ghz in (1, 5, 9, 30):
        if f_ghz == 1:
            row = matrix["ground0_pec" if name == "best" else name]
        else:
            row = verify[name][str(f_ghz)]
        vals = []
        for label in measured:
            f, l, q, _ = measured[label]
            i = int(np.argmin(np.abs(f - f_ghz)))
            vals.append(l[i])
        print(
            f"{name:9} {f_ghz:6d} {row['L_nH']:7.3f} {row['Q']:7.2f} | {vals[0]:7.3f} {vals[1]:7.3f}"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q, s = measured[label]
    axes[0].plot(f, l, color=color, lw=1.0, label=f"measured {label}")
    axes[1].plot(f, q, color=color, lw=1.0, label=f"measured {label}")
f_p = np.array([row[0] for row in palace])
axes[0].plot(f_p, [row[1] for row in palace], "o-", color="C3", label="Palace")
axes[1].plot(f_p, [row[2] for row in palace], "o-", color="C3", label="Palace")
for ax in axes:
    ax.axvline(30, color="k", lw=0.8, ls=":")
    ax.grid(alpha=0.3)
    ax.set(xlabel="frequency (GHz)")
    ax.legend(fontsize=8)
axes[0].set(title="Extracted series inductance", ylabel="L (nH)")
axes[1].set(title="Extracted quality factor", ylabel="Q")
fig.tight_layout()
plt.show()

In [ ]:
# S-parameter comparison at 30 GHz and across the measured band
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    f, l, q, s = measured[label]
    axes[0].plot(
        f, 20 * np.log10(np.abs(s[:, 0, 0])), color=color, lw=1.0, label=f"{label} S11"
    )
    axes[0].plot(
        f,
        20 * np.log10(np.abs(s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
    axes[1].plot(
        f, 20 * np.log10(np.abs(s[:, 0, 0])), color=color, lw=1.0, label=f"{label} S11"
    )
    axes[1].plot(
        f,
        20 * np.log10(np.abs(s[:, 1, 0])),
        color=color,
        lw=1.0,
        ls="--",
        label=f"{label} S21",
    )
for i, f_ghz in enumerate(np.asarray(sp.freq, dtype=float)):
    s = b.s_matrix(sp, i)
    axes[0].plot([f_ghz], [20 * np.log10(abs(s[0, 0]))], "o", color="C3")
    axes[0].plot([f_ghz], [20 * np.log10(abs(s[1, 0]))], "s", color="C3")
axes[0].set(
    title="Measured and Palace S-parameters", xlabel="frequency (GHz)", ylabel="dB"
)
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(alpha=0.3)
axes[1].set(
    title="Measured S-parameters (full band)", xlabel="frequency (GHz)", ylabel="dB"
)
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# All four complex S-parameters, with measured data sampled at Palace frequencies.
# Both measured files and Palace use a 50-ohm reference impedance.
from matplotlib.ticker import MaxNLocator

f_sim = np.asarray(sp.freq, dtype=float)
s_sim = np.stack([b.s_matrix(sp, i) for i in range(len(f_sim))])
assert s_sim.shape == (11, 2, 2)
assert np.all(np.isfinite(s_sim))

s_labels = ((0, 0, "S11"), (1, 0, "S21"), (0, 1, "S12"), (1, 1, "S22"))
comparison = {}
fig, axes = plt.subplots(4, 2, figsize=(12, 12), sharex=True)
for label, color in (("die_x2y7", "C0"), ("die_x5y8", "C1")):
    network = rf.Network(str(b.MEASURED[label]))
    assert np.allclose(network.z0, b.Z0_OHM)
    f_meas = network.f / 1e9
    assert f_meas[0] <= f_sim[0] and f_meas[-1] >= f_sim[-1] - 1e-9
    s_ref = np.empty_like(s_sim)
    for i, j, name in s_labels:
        trace = network.s[:, i, j]
        s_ref[:, i, j] = np.interp(f_sim, f_meas, trace.real) + 1j * np.interp(
            f_sim, f_meas, trace.imag
        )
        delta = s_sim[:, i, j] - s_ref[:, i, j]
        mag_db = lambda s: 20 * np.log10(np.maximum(np.abs(s), 1e-6))
        # Circular phase difference; omit phase where the reference is near a null.
        phase_mask = np.abs(s_ref[:, i, j]) >= 0.03
        phase_error = np.angle(
            s_sim[phase_mask, i, j] / s_ref[phase_mask, i, j], deg=True
        )
        comparison.setdefault(label, {})[name] = {
            "complex_rms": float(np.sqrt(np.mean(np.abs(delta) ** 2))),
            "magnitude_rms_db": float(
                np.sqrt(np.mean((mag_db(s_sim[:, i, j]) - mag_db(s_ref[:, i, j])) ** 2))
            ),
            "phase_rms_deg": float(np.sqrt(np.mean(phase_error**2)))
            if phase_error.size
            else None,
            "phase_points": int(phase_error.size),
        }
        row = [item[2] for item in s_labels].index(name)
        axes[row, 0].plot(
            f_meas, mag_db(trace), color=color, lw=1, label=f"measured {label}"
        )
        axes[row, 1].plot(
            f_meas,
            np.angle(trace, deg=True),
            color=color,
            lw=1,
            label=f"measured {label}",
        )
for row, (i, j, name) in enumerate(s_labels):
    trace = s_sim[:, i, j]
    axes[row, 0].plot(
        f_sim,
        20 * np.log10(np.maximum(np.abs(trace), 1e-6)),
        "o-",
        color="C3",
        ms=3,
        label="Palace",
    )
    axes[row, 1].plot(
        f_sim, np.angle(trace, deg=True), "o-", color="C3", ms=3, label="Palace"
    )
    axes[row, 0].set(ylabel=f"{name} (dB)")
    axes[row, 1].set(ylabel=f"{name} phase (deg)")
    for ax in axes[row]:
        ax.grid(alpha=0.3)
        ax.xaxis.set_major_locator(MaxNLocator(6))
axes[0, 0].legend(fontsize=7)
axes[0, 1].legend(fontsize=7)
axes[-1, 0].set(xlabel="frequency (GHz)")
axes[-1, 1].set(xlabel="frequency (GHz)")
fig.suptitle("Complete two-port comparison: measured .s2p vs Palace")
fig.tight_layout()
plt.show()

print("RMS errors at 11 Palace frequencies (1–50 GHz); phase excludes |Smeas| < 0.03")
for label, rows in comparison.items():
    print(label)
    for name, metrics in rows.items():
        print(
            f"  {name}: complex={metrics['complex_rms']:.3f}, magnitude={metrics['magnitude_rms_db']:.2f} dB, phase={metrics['phase_rms_deg']:.1f} deg ({metrics['phase_points']} points)"
        )

## Experiment matrix

The variants below are exploratory model and port sensitivity runs, not
validated device models. `ground0_pec` was selected after looking at the
measured 1 GHz series-extraction value. Its full S-matrix comparison above
shows the actual disagreement. The `pass` column means only that the selected
extracted inductance was finite and positive.


In [ ]:
import json
from pathlib import Path

matrix = json.loads(
    (
        b.REPO / "nbs" / "data" / "ihp_validation" / "experiments" / "results.json"
    ).read_text()
)
print(f"{'variant':18} {'L 1GHz':>8} {'Q':>7} {'L diff':>8} {'pass':>5}")
for name, row in matrix.items():
    print(
        f"{name:18} {row.get('L_nH', float('nan')):8.3f} {row.get('Q', float('nan')):7.2f} {row.get('L_diff_nH', float('nan')):8.3f} {str(row.get('pass')):>5}"
    )

## Findings and limitations

- The complete S-matrix disagrees with both measured dies in magnitude and
  phase. The table above quantifies each of S11, S21, S12 and S22 at all 11
  Palace frequencies. The model is not validated.
- The 1 GHz `ground0_pec` series-extraction value of 1.921 nH is close to the
  measured 1.911/1.932 nH, but the IHP differential extraction gives about
  0.80 nH for Palace versus about 1.67 nH measured. Low-frequency Q is about
  0.21 versus about 6.6 measured. The series-only match is misleading.
- The outer wall is unconditioned when the absorbing boundary is disabled; it
  should not be called PEC. Port geometry and ground return differ from the
  pinned OpenEMS model. Mesh error Norm is about 0.18 against AdaptiveTol 0.02.
- The 30 GHz point lies near/past a measured resonance, so a single extracted
  inductance there is not a meaningful validation metric. Compare the complex
  response above instead.
